____
### 0. Preamble
This notebook serves to create a Transformer autoregression model to solve the SQL autocomplete problem

____
### 1. Imports

In [3]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [4]:
from autoregression.data.dataprep import N_TOKENS, PAD_ID, EOS_ID, SOS_ID, TOKEN_TO_ID, ID_TO_TOKEN, collate
from autoregression.data.dataprep import SQLSeqDataset

____
### 2. The Transformer approach
- built entirely out of attention instead of recurrence like a RNN
- instead of processing a sequence one step at a time like a RNN, every position in the sequence looks directly at every other position, weighted by learned relevance scores
- a transformer block is made of 2 repeated sub-layers, each wrapped in a residual connection and layer normalization
    1. Multi-head self-attention : lets each token gather information from other tokens.
    2. Position-wise feed-forward network (MLP) : a small 2-layer network applied identically to each position.
- these blocks are stacked for N times to form a transformer, stack more to increase model size and 




#### 2.1 Attention
- the mechanism that lets each position in a sequence decide, dynamically, which other positions to "look at" when building its own representation
- instead of only seeing a fixed neighborhood or only the past through a compressed running state (LSTM's hidden state), evey position looks at the whole sequence and attention decides which part of the sequence to pay extra attention to 
- Every token produces three vectors, via learned linear projections of its embedding:
    1. Query (Q) — "what am I looking for?"
    2. Key (K) — "what do I contain, that others might want?"
    3. Value (V) — "what information do I actually offer, if picked?"
- The attention output of a token is the weighted sum of all value vectors `V`, where the weights come from the **dot product** of that token's query with every other token's key, **scaled** and passed through a **softmax**

$Attention(Q, K, V) = softmax(QK^T / √d_k) · V$

- where: 
    1. $Q_i$ is row `i` of the `Q` tensor, the query vector for whichever token sits at pos `i`
    2. $K_i$ is row `i` of the `K` tensor, the Key vector for whichever token sits at pos `i`
    3. $QK^T$ : the matrix of dot products between every vector inside `Q` and `K`, the similarity matrix
        - Each individual entry `[i, j]` of $QK^T$ is one dot product: $Q_i · K_j$ (a single number).
        - the `[i,j]` entry measures "how well does what pos `i` is looking for $Q_i$ match what position `j` is offering $K_j$"
        - this is becuase dot products answer how close of a match 2 vectors are, dot product = 0 means that they are orthogonal meaning completely different
    4. $√d_k$ : a scaling factor to keep dot products from getting too large
    5. $softmax(...)$ : applied per row (per query position `i`) across all `j`
        - converts raw similarity scores into a probability distribution over j that sums to 1, relative to the other scores in that row — not an absolute similarity measure
        - row `i` = "how token i's attention budget gets split across all positions j"
        - masked entries (future positions under the causal mask, or `<PAD>` positions) are set to -inf beforehand, so they get exactly 0 probability after softmax — they contribute nothing to V
    6. $· V$ : multiplying by Value matrix to gather the information 
        - dot products each value of V with the corresponding value in the similarity matrix after applying `softmax(...)`
        - this computes a soft learned average across every token's Value vector, weighted by relevance, separately for every output dimension.
- attention thus allow a whole sequence to be processed wholesale to find the similarities and draw relationships between tokens without losing information like a RNN would 

#### 2.2 Functionality and Weakness of Attention
1. Multi-head attention
    - runs several attention calculations in parallel with different learned projections
    - model can capture different types of relationships (syntax, long-range dependency, local pattern, etc)
    - the model then simultaneously concatenates and projects the results

2. Causal (masked) self attention:
    -Each attention must only attend to positions before it, not the future - ie the nth position should only be generated when taking into account the tokens from the 1st to the n-1th position
    - This is enforced with a causal mask
    - When computing attention scores, all positions after the current one are set to `-inf` before the softmax, so their weights become 0 

3. Parallel training over every position at once
    - Attention is not sequential like a RNN's hidden state and cam be fed through the model in one forward pass
    - due to causal masking, the model will yield a valid predicted next output at every position simultaneously
    - A sequence of length `T` yields `T-1` supervised training examples in a single pass insted of needing `T-1` seperate forward passes

2. Positional Encoding (Weakness)
    - Attention has no built-in notion of order, treating the sequence as a set
    - To fix this, a positional signal (sinusoidal encoding or learned embedding) is added to each token's input embedding so the model knows where in the sequence each token sits

#### 2.3 Decoder-only Transformer architecture, Generative Pretrained Transformer (GPT) style

- GPT = Generative Pretrained Transformer
    1. Generative: produces new sequences one token at a time, token = SQL block
    2. Pretrained: trained first on a broad next-token-prediction objective, before being fine-tuned or specialized for a specific downstream task
    3. Transformer: the architecture itself

- The original Transformer used 2 separate stacks, an encoder and a decoder
    1. Encoder: reads the entire source sequence at once, no causal masking
        - every position can see every other position, since the whole input is already available
    2. Decoder: generates the target sequence one token at a time, using 2 kinds of attention per layer
        - masked self-attention over what it's generated so far
        - cross-attention that looks back at the encoder's output, so it knows what it's translating

- GPT keeps only the decoder-style stack, and drops cross-attention entirely
    - no separate source sequence to translate from — the model is only ever predicting the next token of the same sequence it's already seen
    - so each layer is just: masked self-attention → feed-forward, no cross-attention block

```md
tokens → token embedding + positional encoding (STEP 1)
       → [N × (masked multi-head self-attention → add & norm → feed-forward → add & norm)] (STEP 2 - 4)
       → final layer norm (STEP 5)
       → linear projection to vocabulary size (STEP 6)
       → softmax → probability distribution over next token (STEP 7)
```

1. token embedding + positional encoding: 
    - turns token ids into vectors, and injects position info
    - position info required since self-attention has no inherent sense of sequence order unlike an LSTM which gets order for free by processing one token at a time

2. masked multi-head self-attention: 
    - computes $softmax(QK^T / √d_k) · V$ across several attention heads in parallel, each potentially learning different relationships between tokens
    - causal masking sets attention scores for future positions to `-inf` before the softmax, so position `i` can only attend to positions `≤ i`

3. add & norm (×2 per layer, around both sub-blocks):
    - *add*: a residual connection, `x + sublayer_output(x)`
        - gives gradients a direct path backward through the network, so each layer only has to learn a correction rather than reconstructing the representation from scratch
    - *norm*: LayerNorm, 
        - rescales activations to a stable mean/variance 
        - keeps training stable across many stacked layers
4. feed-forward
    - a small 2-layer MLP (`emb_dim → ff_dim → emb_dim`) applied independently to each position 
    - gives the model extra per-token processing capacity beyond just mixing information via attention

5. final layer norm
    - one more normalization pass after all N layers, before projecting to vocab

6. linear projection to vocabulary size
    - maps each position's final `emb_dim`-sized vector to `vocab_size` raw logits — one score per possible next token

7. softmax → probability distribution
    - converts logits into probabilities
    - only applied explicitly at generation/sampling time
    - during training, `CrossEntropyLoss` applies softmax (technically log-softmax) internally, so the model itself only ever needs to output raw logits

____
### 3. Implementation

#### 3.1 TransformerCore
- the engine that powers the model

In [5]:
class TransformerCore(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_heads, num_layers,
                 ff_dim, dropout, pad_id, max_len):
        super().__init__()
        self.pad_id = pad_id
        self.max_len = max_len
        self.emb_dim = emb_dim
        self.token_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.pos_emb = nn.Embedding(max_len, emb_dim)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape
        assert T <= self.max_len, f"sequence length {T} exceeds max_len {self.max_len}"
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(positions)         
        x = self.dropout(x)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T).to(input_ids.device)
        key_padding_mask = (input_ids == self.pad_id)
        out = self.encoder(
            x,
            mask=causal_mask,
            src_key_padding_mask=key_padding_mask,
        )
        out = self.ln_f(out)
        logits = self.head(out)
        return logits

##### 3.11 Initialisation
```python
self.pad_id = pad_id
self.max_len = max_len
self.emb_dim = emb_dim
self.token_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
self.pos_emb = nn.Embedding(max_len, emb_dim)
self.dropout = nn.Dropout(dropout)
encoder_layer = nn.TransformerEncoderLayer(
    d_model=emb_dim,
    nhead=num_heads,
    dim_feedforward=ff_dim,
    dropout=dropout,
    batch_first=True,
    norm_first=True)
self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
self.ln_f = nn.LayerNorm(emb_dim)
self.head = nn.Linear(emb_dim, vocab_size)
```
- `self.token_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)` : `nn.Embedding` is a lookup table that maps integer token IDs to dense vectors
    - this removes the need to feed in raw int IDs for each SQL block, removing the implied relationship between SQL blocks due to the numbers being close / far from each other
- `self.pos_emb = nn.Embedding(max_len, emb_dim)` : a seperate embedding table that converts token position to vectors, `max_len` caps how many distinct position vectors exist, cannot embed a position beyond that
    - this gives the model a sense of order, if not the Transformer would just read a sequence as a set of tokens instead of recognising its order
- `self.dropout = nn.Dropout(dropout)` : a separate dropout applied after the LSTM's final output, before the linear head.
    - this is distinct from the LSTM's internal inter-layer dropout, this one hits the final (B, T, hidden_dim) output.
    - this is done to make sure the final layer receives the zeroing treatment as well before being fed into `self.head`
- `encoder_layer = nn.TransformerEncoderLayer(...)` : creating the Encoder layer
    - `d_model=emb_dim` : dimensionality of the vectors flowing through the entire layer.
        - every token's representation at every stage (input, after attention, after feed-forward, output) is of this size
    - `nhead=num_heads` : the number of attention heads to split the computation into
        - e.g. `emb_dim=128, num_heads=4` means that each head operates on 32-dimensional Q/K/V vectors
        - each head computes its own attention pattern independently and gets concatenated back together
    - `dim_feedforward=ff_dim` : the hidden width of the feed-forward sub-block inside the encoder layer
        - expands to a wider dimension, applies non-linearity and projects back down : `d_model → dim_feedforward → d_model`
        - usually set 4x more to give the model extra per-token processing capacity beyond what attentino alone provides
    - `dropout=dropout` : to apply dropout within the encoder layer
    - `batch_first=True` : controls tensor shape convention, `True` makes input/output shape be `(batch, seq_len, features)`
    - `norm_first=True` : controls where LayerNorm sits relative to each sub-block
        - `True` means that data is normalized before feeding into attention/feed-forward, adding residual after
        - `False` means that the sublayers are run first, adding the residual and normalizing after
- `self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)` : creates the encoder by stacking layers on top of each other 
- `self.ln_f = nn.LayerNorm(emb_dim)` : the final layer norm
    - one last cleanup after all N layers
- `self.head = nn.Linear(emb_dim, vocab_size)` : the output projection layer from the embedded dimensitn back to vocabulary space
    - takes `emb_dim`-sized vector and maps it to `vocab_size` raw scores (in the form of logits)
    - one score per possible next token is generated 
    - what CrossEntropyLoss (or a softmax, at generation time) operates on to decide what token comes next
    - turns "internal representation" into "prediction over the vocabulary"

##### 3.12 `forward()`
```python
def forward(self, input_ids):
    B, T = input_ids.shape
    assert T <= self.max_len, f"sequence length {T} exceeds max_len {self.max_len}"
    positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
    x = self.token_emb(input_ids) + self.pos_emb(positions)         
    x = self.dropout(x)
    causal_mask = nn.Transformer.generate_square_subsequent_mask(T).to(input_ids.device)
    key_padding_mask = (input_ids == self.pad_id)
    out = self.encoder(
        x,
        mask=causal_mask,
        src_key_padding_mask=key_padding_mask)
    out = self.ln_f(out)
    logits = self.head(out)
    return logits
```

- `B, T = input_ids.shape` : unpacks batch size and sequence length
- `assert T <= self.max_len, f"sequence length {T} exceeds max_len {self.max_len}"` : troubleshooting
    - to guard against sequences longer than `max_len`
- `positions = torch.arange(T, device=input_ids.device).unsqueeze(0)` : builds the tensor and adds a batch dimenstin so that its shaped `(1, T)` instead of just `(T,)`
- `x = self.token_emb(input_ids) + self.pos_emb(positions)` : implement embeddings
    - the embeddings are added elementwise to form 1 final vector
    - each position's final embedding represents "what token is this" + "where does this token sit in the seq"
- `x = self.dropout(x)` : implement dropout
- `causal_mask = nn.Transformer.generate_square_subsequent_mask(T).to(input_ids.device)` : Builds the `(T, T)` mask where any position further than where the sequence is up to has a value of 0
    - this is added to ${QK^T/√d_k}$ before applying softmax to force future positions to receive exactly zero attention weight
- `key_padding_mask = (input_ids == self.pad_id)` : A boolean mask, per batch item which marks the positions which are `<PAD>`
    - passed seperately from the causal mask because it's data dependent
- `out = self.encoder(x, mask=causal_mask, src_key_padding_mask=key_padding_mask)` : runs `x` through all `num_layers` stacked encoder layers at once
    - internally, every layer applies both masks before every attention softmax so no layer ever lets a position attend to the future or to padding, at any depth in th stack
- `out = self.ln_f(out)` : final normalization
- `logits = self.head(out)` : projection into vocab space, returning a tensor of logit values
    - ready to be compared against `target_ids` via CrossEntropyLoss

#### 3.2 TransformerAgent
- class that owns the TransformerCore 

In [15]:
class TransformerAgent:
    def __init__(self, vocab_size=N_TOKENS, pad_id=PAD_ID, sos_id=SOS_ID, eos_id=EOS_ID,
        emb_dim=128, num_heads=4, num_layers=4, ff_dim=512, dropout=0.1, max_len=64, lr=3e-4,device=None):
        self.pad_id = pad_id
        self.sos_id = sos_id
        self.eos_id = eos_id
        self.max_len = max_len
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = TransformerCore(
            vocab_size=vocab_size,
            emb_dim=emb_dim,
            num_heads=num_heads,
            num_layers=num_layers,
            ff_dim=ff_dim,
            dropout=dropout,
            pad_id=pad_id,
            max_len=max_len,
        ).to(self.device)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)


    def fit(self, training_tensors, validation_tensors, collate_fn,
            epochs=20, batch_size=32, verbose=True):
        train_loader = DataLoader(SQLSeqDataset(training_tensors), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        history = {"training pplx": [], "validation pplx": []}
        for epoch in range(1, epochs + 1):
            train_ppl = self.run_epoch(train_loader, train=True)
            val_ppl = self.evaluate(validation_tensors, collate_fn, batch_size=batch_size)
            history["training pplx"].append(train_ppl)
            history["validation pplx"].append(val_ppl)
            if verbose:
                print(f"epoch {epoch:2d} | training pplx {train_ppl:7.3f} | validation pplx {val_ppl:7.3f}")
        return history

    def run_epoch(self, loader, train):
        self.model.train(mode=train)
        total_loss, total_tokens = 0.0, 0
        context = torch.enable_grad() if train else torch.no_grad()
        with context:
            for input_ids, target_ids in loader:
                input_ids = input_ids.to(self.device)
                target_ids = target_ids.to(self.device)
                if train:
                    self.optimizer.zero_grad()
                logits = self.model(input_ids)
                loss = self.criterion(
                    logits.reshape(-1, logits.size(-1)),
                    target_ids.reshape(-1))
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()
                n_real_tokens = (target_ids != self.pad_id).sum().item()
                total_loss += loss.item() * n_real_tokens
                total_tokens += n_real_tokens
        return math.exp(total_loss / total_tokens)

    def evaluate(self, sequences, collate_fn, batch_size=32, n=1, verbose=False):
        loader = DataLoader(
            SQLSeqDataset(sequences), batch_size=batch_size, shuffle=False,
            collate_fn=collate_fn,
        )
        pplx_lst = []
        for _ in range(0, n):
            pplx = self.run_epoch(loader, train=False)
            if n == 1:
                return pplx
            else:
                pplx_lst.append(pplx)
        avg_pplx = sum(pplx_lst) / len(pplx_lst)
        if verbose:
            print(f"Average perplexity over {n} runs: {round(avg_pplx,2)}")
        return avg_pplx

    @torch.no_grad()
    def sample(self, id_to_token=None, max_len=40, temperature=1.0):
        self.model.eval()
        generated = [self.sos_id]
        for _ in range(max_len):
            input_ids = torch.tensor([generated[-self.max_len:]], device=self.device)  # (1, T)
            logits = self.model(input_ids)                       # (1, T, vocab)
            probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)     # (1, 1)
            token = next_id.item()
            generated.append(token)
            if token == self.eos_id:
                break
        if id_to_token is not None:
            return [id_to_token[i] for i in generated]
        return generated

    def save(self, path):
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path, map_location=self.device))

##### 3.21 Initialisation
```python
self.pad_id = pad_id
self.sos_id = sos_id
self.eos_id = eos_id
self.max_len = max_len
self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
self.model = TransformerCore(
        vocab_size=vocab_size,
        emb_dim=emb_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        ff_dim=ff_dim,
        dropout=dropout,
        pad_id=pad_id,
        max_len=max_len).to(self.device)
self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

```
- `self.model = TransformerCore(...)` : initialises the Transformer core with the input parameters
- `self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)` : registers the model's parameters inside the Adam optimiser
    - this is what allows gradients to flow through all parameters during the training process
- `self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)` : the loss function used for back propagation
    - `ignore_index=pad_id` tells the loss to skip positions where `pad_id` is present
    - this is necessary as a large portion of the sequence might be padding, especially early on

##### 3.22 `run_epoch()`
```python
def run_epoch(self, loader, train):
    self.model.train(mode=train)
    total_loss, total_tokens = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for input_ids, target_ids in loader:
            input_ids = input_ids.to(self.device)
            target_ids = target_ids.to(self.device)
            if train:
                self.optimizer.zero_grad()
            logits = self.model(input_ids)             # (B, T, V) -- no hidden state to unpack
            loss = self.criterion(
                logits.reshape(-1, logits.size(-1)),
                target_ids.reshape(-1))
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            n_real_tokens = (target_ids != self.pad_id).sum().item()
            total_loss += loss.item() * n_real_tokens
            total_tokens += n_real_tokens
    return math.exp(total_loss / total_tokens)
```

- `self.model.train(mode=train)` : sets the model's mode accordingly
    - `train=True` turns the model to training mode `model.train()`
    - `train=False` turns the model to evaluation mode `model.eval()`
- `total_loss, total_tokens = 0.0, 0` : initialising empty values first
- `context = torch.enable_grad() if train else torch.no_grad()` : enables or disables gradients
    - needed to for reusability for training or evaluation
- `with context` : becomes `with torch.enable_grad()` for training and `with torch.no_grad()` for evaluation
- `for input_ids, target_ids in loader`, `input_ids = input_ids.to(self.device)` and `target_ids = target_ids.to(self.device)` : iterates through the `DataLoader` object passed in and moving the tensors to the device for training
- `if train: self.optimizer.zero_grad()` : clears gradients from the previous batch 
- `logits = self.model(input_ids)` : runs the forward pass
    - feeds the input `input_id` from the batch into the Transformer model `self.model`
    - stores the output as `logits`
- `loss = self.criterion(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))` : calculates the loss value using the logits and the target
    - `.reshape` is called to flatten the 2 inputs as they `CrossEntropyLoss` expects 2D input `(N, vocab)` and 1D target `(N, )`
- `if train, loss.backward()` : backpropagation, computes gradients for every parameter
- `torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)` : rescales gradients if their combined norm exceeds 1.0
    - this is to prevent gradient changes from being to large and destabilise training
- `self.optimizer.step()` : applies the update to the weights using the gradients computed
- `n_real_tokens = (target_ids != self.pad_id).sum().item()` : filters out the tokens that are padding
- `total_loss += loss.item() * n_real_tokens` : adds the batch's total loss 
- `total_tokens += n_real_tokens` : adds the number of real tokens for tracking
- `math.exp(total_loss / total_tokens)` 
    - `total_loss/total_tokens` is the epochs avg cross-entropy loss per real token
    - applying `math.exp()` converts it to perplexity, roughly interpretable as "how many tokens the mode was on avg uncertain between", the lower the better 
    - perplexity = 1 means the model was perfectly confident

##### 3.23 `evaluate()`
```python
def evaluate(self, sequences, collate_fn, batch_size=32, n=1, verbose=False):
    loader = DataLoader(
        SQLSeqDataset(sequences), batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn,
    )
    pplx_lst = []
    for _ in range (0,n):
        pplx = self.run_epoch(loader, train=False)
        if n == 1:
            return pplx
        else:
            pplx_lst.append(pplx)
    avg_pplx = sum(pplx_lst)/len(pplx_lst)
    if verbose:
        print(f"Average perplexity over {n} runs: {avg_pplx}")
    return avg_pplx
```
- runs 1 epoch under evaluation mode by default, returning the perplexity
- else, run `n` and return the perplexity over `n` runs

 ##### 3.24 `fit()`
```python
def fit(self, training_tensors, validation_tensors, collate_fn,
    epochs=20, batch_size=32, verbose=True):
    train_loader = DataLoader(SQLSeqDataset(training_tensors), 
        batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    history = {"train_ppl": [], "val_ppl": []}
    for epoch in range(1, epochs + 1):
        train_ppl = self.run_epoch(train_loader, train=True)
        val_ppl = self.evaluate(validation_tensors, collate_fn, batch_size=batch_size)
        history["train_ppl"].append(train_ppl)
        history["val_ppl"].append(val_ppl)
        if verbose:
            print(f"epoch {epoch:2d} | train ppl {train_ppl:7.3f} | val ppl {val_ppl:7.3f}")
    return history
```

- `train_loader = DataLoader(SQLSeqDataset(training_tensors), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)` : creating the `DataLoader` object
    - `SQLSeqDataset(training_tensors)` creates the dataset object and sends it into the `Dataloader` object directly 
- `history = {"train_ppl": [], "val_ppl": []}` : initialise a dictionary to store values
    - `train_ppl` stores the perplexity value for the training set and `val_ppl` stores the perplexity value for the validation set, throughout the epochs
- `for epoch in range(1, epochs + 1):` iterate for the number of epochs indicated
- `train_ppl = self.run_epoch(train_loader, train=True)` : runs 1 epoch under training mode, collecting the peplexity value
- `val_ppl = self.evaluate(validation_tensors, collate_fn, batch_size=batch_size)` : runs 1 epoch under evaluations mode, measuring how well how the model generalizes right now
- `history["train_ppl"].append(train_ppl)` and `history["val_ppl"].append(val_ppl)` : store perplexity values for the epoch
- `if verbose: ...` : printing for logging

##### 3.35 `sample()`
```python
@torch.no_grad()
def sample(self, id_to_token=None, max_len=40, temperature=1.0):
    self.model.eval()
    generated = [self.sos_id]
    for _ in range(max_len):
        input_ids = torch.tensor([generated[-self.max_len:]], device=self.device)  # (1, T)
        logits = self.model(input_ids)                       # (1, T, vocab)
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)     # (1, 1)
        token = next_id.item()
        generated.append(token)
        if token == self.eos_id:
            break
    if id_to_token is not None:
        return [id_to_token[i] for i in generated]
    return generated
```
- `@torch.no_grad()` : ensures the function runs without computing gradients so the model weights don't change
- `self.model.eval()` : switches the model to evaluation mode
- `generated = [self.sos_id]` : initialises the output sequence with the start-of-sequence token id
    - unlike the LSTM version, there's no `hidden = None` line here — the Transformer has no recurrent state to initialise, since attention gives every position direct access to earlier positions rather than relying on a carried-forward summary
- `for _ in range(max_len):` : generates up to `max_len` new tokens, one at a time
- `input_ids = torch.tensor([generated[-self.max_len:]], device=self.device)` : rebuilds the input tensor from the **entire sequence generated so far**, not just the newest token
    - this is the key difference from the LSTM's `sample()`, which fed only `next_id` each step and let `hidden` carry the past
    - the Transformer has no equivalent to `hidden`, so on every iteration it has to re-supply the whole prefix for self-attention to look back over
    - `[-self.max_len:]` truncates to the last `max_len` tokens, since `TransformerCore.forward` asserts the input can't exceed `max_len` (a hard ceiling from the fixed-size `pos_emb` table) — the LSTM has no equivalent constraint, since recurrence can in principle run over any length
- `logits = self.model(input_ids)` : runs the forward pass over the current prefix
    - shape `(1, T, vocab)`, where `T` grows by one each iteration — different from the LSTM's fixed `(1, 1, vocab)`, since here the whole growing sequence is passed in every time, not just one token
- `probs = torch.softmax(logits[:, -1] / temperature, dim=-1)` : takes only the logits at the **last position** (the prediction for what comes after everything generated so far) and converts to probabilities
    - `temperature` scales the logits before softmax: `>1` flattens the distribution (more random/diverse sampling), `<1` sharpens it (more confident/greedy sampling)
- `next_id = torch.multinomial(probs, num_samples=1)` : samples one token id from that probability distribution, rather than always picking the highest-probability token (which would make generation deterministic)
- `token = next_id.item()` and `generated.append(token)` : extracts the plain integer and appends it to the running sequence
- `if token == self.eos_id: break` : stops early once the model generates the end-of-sequence token, rather than always running the full `max_len` steps
- note there's no `input_id = next_id` line at the end of the loop like in the LSTM version — since state isn't carried forward, the next iteration's `input_ids` line simply rebuilds itself from the now-one-token-longer `generated` list
- `if id_to_token is not None: return [id_to_token[i] for i in generated]` : optionally converts ids back into readable token names before returning

**cost tradeoff**: 
- because the whole prefix is re-fed and re-processed by self-attention every step, generation cost grows with sequence length (step `t` costs `O(t)` instead of the LSTM's `O(1)` per step). 
- A proper implementation would cache each layer's key/value projections for already-seen positions (KV-caching) so only the new token needs fresh attention computation 
- this is what makes production LLM inference fast, and is worth adding once generating longer sequences than `max_len=40`

____
### 4. Training and Evauating the LSTM model
1. Import training and evaluation data


In [7]:
from autoregression.data.datasplit import training_tensors, validation_tensors

Audit OK: all held-out variants are safe (no token is solely sourced by a held-out variant).
train:1277 problems
interp_val: 67 problems
ood[select]:  336 problems (only 'select' is novel)
ood[where]:  384 problems (only 'where' is novel)
ood[groupby]:  336 problems (only 'groupby' is novel)
ood[orderby]:  448 problems (only 'orderby' is novel)
ood[limit]:  672 problems (only 'limit' is novel)
ood_compound: 1880 problems (2+ knobs novel at once)

  select: counts per in-distribution value = {0: 336, 1: 336, 2: 336, 3: 336}  (balanced)
  join: counts per in-distribution value = {0: 672, 1: 672}  (balanced)
  where: counts per in-distribution value = {0: 192, 1: 192, 2: 192, 3: 192, 4: 192, 5: 192}  (balanced)
  groupby: counts per in-distribution value = {0: 336, 2: 336, 3: 336, 4: 336}  (balanced)
  orderby: counts per in-distribution value = {0: 448, 2: 448, 3: 448}  (balanced)
  limit: counts per in-distribution value = {0: 672, 2: 672}  (balanced)


2. Instantiate instance of `RNNAgent`

In [16]:
transformer = TransformerAgent(
    emb_dim=128,
    num_heads=4,
    num_layers=4,
    ff_dim=512,
    dropout=0.1,
    max_len=64,
    lr=3e-4)

C:\Users\xuan2\AppData\Local\Temp\ipykernel_34460\478094372.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


3. Train and evaluate

In [12]:
data = transformer.fit(training_tensors, validation_tensors, collate_fn=collate, epochs=500)

c:\Users\xuan2\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\nn\modules\transformer.py:429: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


epoch  1 | training pplx   6.484 | validation pplx   2.367
epoch  2 | training pplx   2.130 | validation pplx   1.757
epoch  3 | training pplx   1.689 | validation pplx   1.529
epoch  4 | training pplx   1.539 | validation pplx   1.483
epoch  5 | training pplx   1.490 | validation pplx   1.446
epoch  6 | training pplx   1.459 | validation pplx   1.431
epoch  7 | training pplx   1.443 | validation pplx   1.414
epoch  8 | training pplx   1.430 | validation pplx   1.407
epoch  9 | training pplx   1.422 | validation pplx   1.412
epoch 10 | training pplx   1.419 | validation pplx   1.402
epoch 11 | training pplx   1.413 | validation pplx   1.398
epoch 12 | training pplx   1.409 | validation pplx   1.411
epoch 13 | training pplx   1.410 | validation pplx   1.396
epoch 14 | training pplx   1.407 | validation pplx   1.391
epoch 15 | training pplx   1.405 | validation pplx   1.391
epoch 16 | training pplx   1.402 | validation pplx   1.393
epoch 17 | training pplx   1.399 | validation pplx   1.3

In [19]:
val_ppl = transformer.evaluate(validation_tensors, collate_fn=collate, n=200, verbose=True)

c:\Users\xuan2\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\nn\modules\transformer.py:429: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


Average perplexity over 200 runs: 1.48


- an average perplexity value of `1.48` means that the model is on average choosing between 1.48 tokens over 200 runs, meaning that it is mostly certain when choosing the next block 

4. Saving to `.pth` file

In [14]:
transformer.save("trans.pth")

- all classes and processes created in this notebook is stored within the files `transformer.py` and `traintrans.py` 